In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import csv
import random
import time

from PIL import Image, ImageDraw
from matplotlib import cm
from scipy.optimize import curve_fit
from scipy.stats import chisquare
from reconstructionClass import Reconstruction as recon

In [ ]:
## FIMS Fit ##
reconstructionInfo = {
    'Gain': 60,
    'Avalanche Sigma': 20,
    'Hole Pitch': 55,
    'Pixel Pitch': 55,
    'Standoff': 30,
    'Signal Decay Rate': .01,
    'Signal Threshold': 10,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
fimsPlot = recon(reconstructionInfo).reconstructFIMS()

In [ ]:
## BEAST Fit ##
#TODO: replace values with proper BEAST values (these are Migdal)
reconstructionInfo = {
    'Gain': 1000,
    'Avalanche Sigma': 300,
    'Hole Pitch': 60,
    'Pixel Pitch': 83,
    'Standoff': 50,
    'Signal Decay Rate': 5000,
    'Signal Threshold': 200,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
beastPlot = recon(reconstructionInfo).reconstructBEAST()

In [ ]:
## Migdal Experiment Fit ##
reconstructionInfo = {
    'Gain': 1000,
    'Avalanche Sigma': 300,
    'Hole Pitch': 60,
    'Pixel Pitch': 83,
    'Standoff': 3000,
    'Signal Decay Rate': 5000,
    'Signal Threshold': 200,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
migdalPlot = recon(reconstructionInfo).reconstructMigdal()

In [ ]:
## GridPix Fit ##
reconstructionInfo = {
    'Gain': 2500,
    'Avalanche Sigma': 500,
    'Hole Pitch': 55,
    'Pixel Pitch': 55,
    'Standoff': 55,
    'Signal Decay Rate': 500,
    'Signal Threshold': 800,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
gridpixPlot = recon(reconstructionInfo).reconstructGridPix()

In [ ]:
## Testing Area ##
## Information needed for an individual event reconstruction ##
reconstructionInfo = {
    'Gain': 100,
    'Avalanche Sigma': 1,
    'Hole Pitch': 55,
    'Pixel Pitch': 55,
    'Standoff': 55,
    'Signal Decay Rate': 500,
    'Signal Threshold': 20,
    'File Location': '../Data/Sven_Recoil_Data.root',
    'Tree Name': 'tree_3d_noquant_before_drift'
}
reconstruction = recon(reconstructionInfo)

## Extract and calculate relevant data ##
# discretization values
pitch = reconstruction.reconInfo['Hole Pitch']
pixPitch = reconstruction.reconInfo['Pixel Pitch']
timeRez = reconstruction.timeRez

# Diffusion values
transDif = reconstruction.transDifCoef*math.sqrt(reconstruction.initialDriftDistance)
lonDif = reconstruction.lonDifCoef*math.sqrt(reconstruction.initialDriftDistance)
secondTransDif = reconstruction.transDifCoef*math.sqrt(reconstruction.reconInfo['Standoff'])
secondLonDif = reconstruction.lonDifCoef*math.sqrt(reconstruction.reconInfo['Standoff'])

# Apply Gaussian smear to approximate diffusion
smearData = reconstruction.diffuseData(reconstruction.rawData, (transDif, transDif, lonDif))

# Discretize data to approximate falling into grid holes
discreteDataFrame = reconstruction.discretizeData(smearData, (pitch, pitch, 0))

# Approximate avalanche
newElectrons = reconstruction.approximateGain(discreteDataFrame)
    
avalData = reconstruction.diffuseData(newElectrons, (secondTransDif, secondTransDif, secondLonDif))

# Discretize data to approximate pixels readout
padData = reconstruction.discretizeData(avalData, (pitch, pitch, timeRez))

# Approximate Signal Readout
readoutData = reconstruction.approximateReadout(padData)

In [ ]:
# Generate random data
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Create scatter plot with color mapping
scatter = ax.scatter(xPlot, yPlot, z, c=c, cmap='viridis')

# Add color bar to indicate the scale of 'c'
fig.colorbar(scatter, ax=ax)

# Set axis labels
ax.set_xlabel('X Axis')
ax.set_ylabel('Y Axis')
ax.set_zlabel('Z Axis')

plt.show()